In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/saved-model-and-dataset/val_dataset/val_dataset/state.json
/kaggle/input/saved-model-and-dataset/val_dataset/val_dataset/dataset_info.json
/kaggle/input/saved-model-and-dataset/val_dataset/val_dataset/data-00000-of-00001.arrow
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/config.json
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/merges.txt
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/vocab.json
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/tokenizer_config.json
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/model.safetensors
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/special_tokens_map.json
/kaggle/input/saved-model-and-dataset/bart_summarizes_final/generation_config.json


In [2]:
!pip install --upgrade pip

In [ ]:
!pip install pyarrow==16.1.0
!pip install datasets==2.20.0

In [ ]:
!pip install transformers==4.41.2
!pip install evaluate==0.4.1

In [ ]:
!pip install nltk==3.8.1

In [ ]:
!pip install rouge_score

In [ ]:
!pip install transformers==4.46.3
!pip install peft==0.11.1

In [ ]:
!pip uninstall -y transformers peft datasets pyarrow
!pip uninstall -y transformers peft datasets pyarrow

In [3]:
!pip install pyarrow==16.1.0
!pip install datasets==2.20.0
!pip install transformers==4.36.2
!pip install peft==0.8.2
!pip install evaluate==0.4.1
!pip install nltk==3.8.1
!pip install rouge_score


In [ ]:
!pip install -U accelerate==0.34.2 transformers==4.44.2

In [ ]:
!pip install --upgrade accelerate==0.33.0 transformers==4.41.2

In [ ]:
!pip install --upgrade fsspec datasets

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()



In [ ]:
!nvidia-smi

In [23]:
import os, json, math, random, time, shutil
import pandas as pd
import numpy as np
import torch
import transformers
from datasets import load_dataset
import re
from transformers import pipeline
import nltk
nltk.download("punkt")
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from transformers import BartTokenizer
from datasets import load_from_disk
from transformers import (BartForConditionalGeneration, Trainer,
                          TrainingArguments, BartTokenizer,
                          Seq2SeqTrainer, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments
                          )
import evaluate
from datasets import load_metric
from rouge_score import rouge_scorer
from torch.utils.data import DataLoader
torch.cuda.empty_cache()

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Validationset matrices calculate using trained model and saved val dataset

In [5]:
model_path = "/kaggle/input/saved-model-and-dataset/bart_summarizes_final"
val_path = "/kaggle/input/saved-model-and-dataset/val_dataset/val_dataset"
val_dataset = load_from_disk(val_path)

In [6]:
tokenizer = BartTokenizer.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path).to("cuda")

In [12]:
val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [16]:
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    r1, r2, rL = [], [], []
    for pred, ref in zip(decoded_preds, decoded_labels):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rL.append(scores["rougeL"].fmeasure)

    result = {
        "rouge1": float(np.mean(r1) * 100),
        "rouge2": float(np.mean(r2) * 100),
        "rougeL": float(np.mean(rL) * 100),
    }
    gen_lens = [np.count_nonzero(p != tokenizer.pad_token_id) for p in predictions]
    result["gen_len"] = float(np.mean(gen_lens))

    return result


In [19]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",    
    return_tensors="pt"
)

In [20]:
subset = val_dataset.select(range(300))
loader = DataLoader(subset, batch_size=4, collate_fn=data_collator)

In [22]:
device = "cuda"
model.eval()

all_preds = []
all_labels = []

for batch in loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"] 

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=1,
            max_length=150
        )

    all_preds.extend(outputs.cpu().numpy())
    all_labels.extend(labels.numpy())

metrics = compute_metrics((np.array(all_preds), np.array(all_labels)))
print(metrics)


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:430: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


{'rouge1': 39.029682606711916, 'rouge2': 15.720115371266575, 'rougeL': 24.159790349189777, 'gen_len': 148.15333333333334}


In [24]:
df = pd.DataFrame([metrics])
df

,rouge1,rouge2,rougeL,gen_len
0,39.029683,15.720115,24.15979,148.153333


In [26]:
csv_path = "/kaggle/working/metrics_report_valset.csv"
df.to_csv(csv_path, index=False)
